In [6]:
# IMPORTS AND DISPLAY SETTINGS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display all DataFrame columns when inspecting results
pd.set_option("display.max_columns", None)

In [7]:
user_artists = pd.read_csv("user_artists.dat", sep="\t")

artists = pd.read_csv("artists.dat", sep="\t")

user_friends = pd.read_csv("user_friends.dat", sep="\t")

tags = pd.read_csv("tags.dat", sep="\t", encoding='latin1')

user_taggedartists = pd.read_csv("user_taggedartists.dat",sep="\t")

In [8]:
print("Number of unique users:")
print(user_artists["userID"].nunique())

print()

print("Number of unique artists:")
print(user_artists["artistID"].nunique())

print()

print("Number of listening interactions:")
print(len(user_artists))

print()

print("Number of directed friendship records:")
print(len(user_friends))

print()

print("Number of unique tags:")
print(tags["tagValue"].nunique())

Number of unique users:
1892

Number of unique artists:
17632

Number of listening interactions:
92834

Number of directed friendship records:
25434

Number of unique tags:
11946


In [9]:
num_users = user_artists.userID.nunique()
num_artists = user_artists.artistID.nunique()

possible_interactions = num_users * num_artists

actual_interactions = len(user_artists)

sparsity = 1 - (actual_interactions / possible_interactions)

print(f"Users: {num_users}")
print(f"Artists: {num_artists}")
print(f"Possible interactions: {possible_interactions:,}")
print(f"Actual interactions: {actual_interactions:,}")
print(f"Sparsity: {sparsity:.4f}")

Users: 1892
Artists: 17632
Possible interactions: 33,359,744
Actual interactions: 92,834
Sparsity: 0.9972


In [10]:
!pip install torch torch-geometric scipy scikit-learn pandas numpy tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00


In [11]:
# LIGHTGCN PREPROCESSING
# Data Loading, Train/Validation/Test Split,
# Graph Construction, Negative Sampling, BPR Dataset


import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data


# LOAD DATASET


ratings = pd.read_csv(
    "user_artists.dat",
    sep="\t"
)

print("=" * 60)
print("DATASET")
print("=" * 60)

print(f"Total Interactions : {len(ratings)}")
print(f"Unique Users       : {ratings.userID.nunique()}")
print(f"Unique Artists     : {ratings.artistID.nunique()}")


# ENCODE USERS AND ARTISTS


user_mapping = {
    user_id: idx
    for idx, user_id in enumerate(ratings["userID"].unique())
}

artist_mapping = {
    artist_id: idx
    for idx, artist_id in enumerate(ratings["artistID"].unique())
}

ratings["user"] = ratings["userID"].map(user_mapping)
ratings["artist"] = ratings["artistID"].map(artist_mapping)

num_users = len(user_mapping)
num_items = len(artist_mapping)

print("\nEncoded IDs")
print(f"Users   : {num_users}")
print(f"Artists : {num_items}")


# TRAIN / VALIDATION / TEST SPLIT


train_list = []
val_list = []
test_list = []

for user in ratings["user"].unique():

    user_df = ratings[
    ratings["user"] == user
]

    if len(user_df) < 3:

        train_list.append(user_df)
        val_list.append(pd.DataFrame(columns=user_df.columns))
        test_list.append(pd.DataFrame(columns=user_df.columns))
        continue

    train, temp = train_test_split(
        user_df,
        test_size=0.20,
        random_state=42
    )

    if len(temp) < 2:

        train_list.append(train)
        val_list.append(temp)
        test_list.append(pd.DataFrame(columns=user_df.columns))

    else:

        val, test = train_test_split(
            temp,
            test_size=0.50,
            random_state=42
        )

        train_list.append(train)
        val_list.append(val)
        test_list.append(test)

train_df = pd.concat(train_list, ignore_index=True)
val_df = pd.concat(val_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)

print("\n" + "=" * 60)
print("DATA SPLIT")
print("=" * 60)

print(f"Training Interactions   : {len(train_df)}")
print(f"Validation Interactions : {len(val_df)}")
print(f"Testing Interactions    : {len(test_df)}")


# GRAPH CONSTRUCTION


user_nodes = train_df["user"].values
item_nodes = train_df["artist"].values + num_users

edge_index = np.vstack([
    np.concatenate([user_nodes, item_nodes]),
    np.concatenate([item_nodes, user_nodes])
])

edge_index = torch.tensor(
    edge_index,
    dtype=torch.long
)

graph = Data(edge_index=edge_index)
graph.num_nodes = num_users + num_items

print("\n" + "=" * 60)
print("GRAPH SUMMARY")
print("=" * 60)

print(f"Total Nodes : {graph.num_nodes}")
print(f"Total Edges : {graph.edge_index.shape[1]}")
print(f"Users       : {num_users}")
print(f"Artists     : {num_items}")


# POSITIVE ITEMS FOR EACH USER


user_pos_items = {}

for row in train_df.itertuples():

    user = row.user
    item = row.artist

    if user not in user_pos_items:
        user_pos_items[user] = set()

    user_pos_items[user].add(item)

print("\nPositive Item Dictionary Created")


# NEGATIVE SAMPLING


def sample_negative(user):

    positives = user_pos_items.get(user, set())

    while True:

        neg_item = random.randint(0, num_items - 1)

        if neg_item not in positives:
            return neg_item


# BPR DATASET


class BPRDataset(Dataset):

    def __init__(self, train_df):

        self.users = train_df["user"].values
        self.pos_items = train_df["artist"].values

    def __len__(self):
        return len(self.users)

    def __getitem__(self, index):

        user = self.users[index]
        pos_item = self.pos_items[index]
        neg_item = sample_negative(user)

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(pos_item, dtype=torch.long),
            torch.tensor(neg_item, dtype=torch.long)
        )


# DATALOADER


train_dataset = BPRDataset(train_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True
)

print("\n" + "=" * 60)
print("DATALOADER")
print("=" * 60)

print(f"Number of Batches : {len(train_loader)}")

# SANITY CHECK


users, pos_items, neg_items = next(iter(train_loader))

print("\nBatch Shapes")

print("Users          :", users.shape)
print("Positive Items :", pos_items.shape)
print("Negative Items :", neg_items.shape)

print("\nExample Training Triple")

print("User     :", users[0].item())
print("Positive :", pos_items[0].item())
print("Negative :", neg_items[0].item())

DATASET
Total Interactions : 92834
Unique Users       : 1892
Unique Artists     : 17632

Encoded IDs
Users   : 1892
Artists : 17632

DATA SPLIT
Training Interactions   : 74250
Validation Interactions : 9282
Testing Interactions    : 9302

GRAPH SUMMARY
Total Nodes : 19524
Total Edges : 148500
Users       : 1892
Artists     : 17632

Positive Item Dictionary Created

DATALOADER
Number of Batches : 37

Batch Shapes
Users          : torch.Size([2048])
Positive Items : torch.Size([2048])
Negative Items : torch.Size([2048])

Example Training Triple
User     : 407
Positive : 915
Negative : 1538


In [12]:
# LIGHTGCN MODEL


import torch
import torch.nn as nn
from torch_geometric.nn import LGConv


class LightGCN(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=128,
        num_layers=2
    ):
        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers


        # Learnable user embeddings

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )


        # Learnable artist embeddings

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        # Xavier Initialization
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)


        # LightGCN propagation layers

        self.convs = nn.ModuleList(
            [LGConv() for _ in range(num_layers)]
        )

    def forward(self, edge_index):

        # Initial embeddings
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight

        x = torch.cat([users_emb, items_emb], dim=0)

        # Store embeddings from every layer
        embeddings = [x]

        # Graph propagation
        for conv in self.convs:

            x = conv(x, edge_index)

            embeddings.append(x)

        # Average embeddings from all layers
        embeddings = torch.stack(embeddings, dim=1)

        final_embeddings = embeddings.mean(dim=1)

        user_final = final_embeddings[:self.num_users]
        item_final = final_embeddings[self.num_users:]

        return user_final, item_final

In [13]:
# EXPERIMENT D2 - CLEAN SOCIAL LIGHTGCN
# Social branch contains ONLY propagated friendship information


import torch
import torch.nn as nn
from torch_geometric.nn import LGConv


class SocialLightGCN(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=128,
        interaction_layers=2,
        social_layers=1,
        alpha=0.05
    ):

        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim

        self.interaction_layers = interaction_layers
        self.social_layers = social_layers

        self.alpha = alpha


        # Trainable user/item embeddings


        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        nn.init.xavier_uniform_(
            self.user_embedding.weight
        )

        nn.init.xavier_uniform_(
            self.item_embedding.weight
        )


        # Original user-artist LightGCN branch


        self.interaction_convs = nn.ModuleList(
            [
                LGConv()
                for _ in range(interaction_layers)
            ]
        )


        # User-user friendship branch


        self.social_convs = nn.ModuleList(
            [
                LGConv()
                for _ in range(social_layers)
            ]
        )

    def forward(
        self,
        interaction_edge_index,
        social_edge_index
    ):

        users_initial = self.user_embedding.weight
        items_initial = self.item_embedding.weight


        # 1. ORIGINAL LIGHTGCN INTERACTION BRANCH


        interaction_x = torch.cat(
            [
                users_initial,
                items_initial
            ],
            dim=0
        )

        interaction_embeddings = [
            interaction_x
        ]

        for conv in self.interaction_convs:

            interaction_x = conv(
                interaction_x,
                interaction_edge_index
            )

            interaction_embeddings.append(
                interaction_x
            )

        interaction_embeddings = torch.stack(
            interaction_embeddings,
            dim=1
        ).mean(dim=1)

        interaction_user = (
            interaction_embeddings[
                :self.num_users
            ]
        )

        interaction_item = (
            interaction_embeddings[
                self.num_users:
            ]
        )


        # 2. SOCIAL BRANCH

        social_x = users_initial

        for conv in self.social_convs:

            social_x = conv(
                social_x,
                social_edge_index
            )

        social_user = social_x


        # 3. FUSION


        final_user = (
            interaction_user
            + self.alpha * social_user
        ) / (1.0 + self.alpha)

        # Artist representation remains from original LightGCN
        final_item = interaction_item

        return final_user, final_item

In [14]:
# PREFERENCE-AWARE SOCIAL LIGHTGCN (Experiment D3)
# Social branch weighted by friend listening similarity

import torch
import torch.nn as nn
from torch_geometric.nn import LGConv


class PreferenceAwareSocialLightGCN(nn.Module):

    def __init__(self, num_users, num_items, embedding_dim=128,
                 interaction_layers=2, social_layers=1, alpha=0.05):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.alpha = alpha

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

        self.interaction_convs = nn.ModuleList(
            [LGConv() for _ in range(interaction_layers)]
        )
        self.social_convs = nn.ModuleList(
            [LGConv() for _ in range(social_layers)]
        )

    def forward(self, interaction_edge_index,
                social_edge_index, social_edge_weight):

        users_initial = self.user_embedding.weight
        items_initial = self.item_embedding.weight

        # Interaction branch (standard LightGCN)
        x = torch.cat([users_initial, items_initial], dim=0)
        embs = [x]
        for conv in self.interaction_convs:
            x = conv(x, interaction_edge_index)
            embs.append(x)
        final = torch.stack(embs, dim=1).mean(dim=1)

        interaction_user = final[:self.num_users]
        interaction_item = final[self.num_users:]

        # Preference-weighted social branch
        social_x = users_initial
        for conv in self.social_convs:
            social_x = conv(
                social_x,
                social_edge_index,
                edge_weight=social_edge_weight
            )

        final_user = (
            interaction_user + self.alpha * social_x
        ) / (1.0 + self.alpha)

        return final_user, interaction_item


print("PreferenceAwareSocialLightGCN defined")

PreferenceAwareSocialLightGCN defined


In [15]:
# EXPERIMENT D - STEP 1
# BUILD USER-USER SOCIAL GRAPH


import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data


# Load friendship data


friends_df = pd.read_csv(
    "user_friends.dat",
    sep="\t"
)

print("=" * 60)
print("RAW FRIENDSHIP DATA")
print("=" * 60)

print(friends_df.head())
print("\nShape:", friends_df.shape)
print("\nColumns:", friends_df.columns.tolist())

RAW FRIENDSHIP DATA
   userID  friendID
0       2       275
1       2       428
2       2       515
3       2       761
4       2       831

Shape: (25434, 2)

Columns: ['userID', 'friendID']


In [16]:
# EXPERIMENT D - STEP 2
# MAP RAW USER IDs TO LIGHTGCN USER INDICES

social_df = friends_df.copy()

# Use EXACT SAME user mapping as the original LightGCN

social_df["user"] = social_df["userID"].map(user_mapping)

social_df["friend"] = social_df["friendID"].map(user_mapping)


# Remove friendships where either user is not represented
# in the recommendation dataset

social_df = social_df.dropna(
    subset=["user", "friend"]
).copy()

social_df["user"] = social_df["user"].astype(int)
social_df["friend"] = social_df["friend"].astype(int)


# Remove self-connections if any


social_df = social_df[
    social_df["user"] != social_df["friend"]
]


# Remove duplicate relationships


social_df = social_df.drop_duplicates(
    subset=["user", "friend"]
)

print("=" * 60)
print("ENCODED SOCIAL DATA")
print("=" * 60)

print(social_df.head())

print("\nValid friendship rows :", len(social_df))
print("Users with friendships:",
      social_df["user"].nunique())

print("\nEncoded user range:")
print(
    social_df[["user", "friend"]]
    .min()
)

print(
    social_df[["user", "friend"]]
    .max()
)

ENCODED SOCIAL DATA
   userID  friendID  user  friend
0       2       275     0     257
1       2       428     0     400
2       2       515     0     482
3       2       761     0     709
4       2       831     0     772

Valid friendship rows : 25434
Users with friendships: 1892

Encoded user range:
user      0
friend    0
dtype: int64
user      1891
friend    1891
dtype: int64


In [17]:
# EXPERIMENT D - STEP 3
# CHECK FRIENDSHIP DIRECTIONALITY


friend_pairs = set(
    zip(
        social_df["user"],
        social_df["friend"]
    )
)

reverse_count = sum(
    (friend, user) in friend_pairs
    for user, friend in friend_pairs
)

total_pairs = len(friend_pairs)

print("=" * 60)
print("FRIENDSHIP SYMMETRY CHECK")
print("=" * 60)

print("Total directed pairs :", total_pairs)
print("Pairs with reverse   :", reverse_count)

if total_pairs > 0:

    symmetry_ratio = reverse_count / total_pairs

    print(
        f"Symmetry ratio       : "
        f"{symmetry_ratio:.4f}"
    )

FRIENDSHIP SYMMETRY CHECK
Total directed pairs : 25434
Pairs with reverse   : 25434
Symmetry ratio       : 1.0000


In [18]:
# EXPERIMENT D - STEP 4
# CREATE UNDIRECTED SOCIAL EDGE INDEX


# Set computation device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Move interaction graph to the same device
graph = graph.to(device)


social_edges = np.vstack([
    social_df["user"].values,
    social_df["friend"].values
])

social_edges = torch.tensor(
    social_edges,
    dtype=torch.long
)

# Reverse direction


reverse_social_edges = social_edges.flip(0)

social_edge_index = torch.cat(
    [
        social_edges,
        reverse_social_edges
    ],
    dim=1
)


# Remove duplicate edges


social_edge_index = torch.unique(
    social_edge_index,
    dim=1
)

social_edge_index = social_edge_index.to(device)

print("=" * 60)
print("SOCIAL GRAPH")
print("=" * 60)

print(
    "Social edge index shape:",
    social_edge_index.shape
)

print(
    "Number of users:",
    num_users
)

print(
    "Minimum node ID:",
    social_edge_index.min().item()
)

print(
    "Maximum node ID:",
    social_edge_index.max().item()
)

SOCIAL GRAPH
Social edge index shape: torch.Size([2, 25434])
Number of users: 1892
Minimum node ID: 0
Maximum node ID: 1891


In [19]:
# EXPERIMENT D3 - STEP 1
# BUILD TRAINING USER-ARTIST MATRIX

from scipy.sparse import csr_matrix
import numpy as np
import pandas as pd
import torch


# Build sparse user-item matrix using TRAIN ONLY
train_user_item = csr_matrix(
    (
        train_df["weight"].astype(float).values,
        (
            train_df["user"].values,
            train_df["artist"].values
        )
    ),
    shape=(num_users, num_items)
)

print("=" * 60)
print("TRAIN USER-ITEM MATRIX")
print("=" * 60)

print("Shape:", train_user_item.shape)
print("Non-zero interactions:", train_user_item.nnz)

TRAIN USER-ITEM MATRIX
Shape: (1892, 17632)
Non-zero interactions: 74250


In [20]:
# EXPERIMENT D3 - STEP 2
# COSINE SIMILARITY FOR REAL FRIENDSHIP EDGES

from sklearn.preprocessing import normalize


# L2-normalise each user's listening vector
normalized_train = normalize(
    train_user_item,
    norm="l2",
    axis=1
)


# social_edge_index already contains:
# row 0 = source user
# row 1 = friend user

social_sources = (
    social_edge_index[0]
    .detach()
    .cpu()
    .numpy()
)

social_targets = (
    social_edge_index[1]
    .detach()
    .cpu()
    .numpy()
)


friend_similarity = []


for u, v in zip(
    social_sources,
    social_targets
):

    # Cosine similarity because vectors
    # have already been L2-normalised

    similarity = (
        normalized_train[u]
        .multiply(normalized_train[v])
        .sum()
    )

    friend_similarity.append(
        float(similarity)
    )


friend_similarity = np.array(
    friend_similarity,
    dtype=np.float32
)


print("=" * 60)
print("FRIEND PREFERENCE SIMILARITY")
print("=" * 60)

print(
    "Number of friendship edges:",
    len(friend_similarity)
)

print(
    "Minimum similarity:",
    friend_similarity.min()
)

print(
    "Maximum similarity:",
    friend_similarity.max()
)

print(
    "Mean similarity:",
    friend_similarity.mean()
)

print(
    "Median similarity:",
    np.median(friend_similarity)
)

print(
    "Zero similarity edges:",
    np.sum(friend_similarity == 0)
)

FRIEND PREFERENCE SIMILARITY
Number of friendship edges: 25434
Minimum similarity: 0.0
Maximum similarity: 0.99920535
Mean similarity: 0.20653538
Median similarity: 0.11452305
Zero similarity edges: 2466


In [21]:
# EXPERIMENT D3 - STEP 4
# CREATE SOCIAL EDGE WEIGHTS


social_edge_weight = torch.tensor(
    friend_similarity,
    dtype=torch.float32,
    device=device
)

print("=" * 60)
print("WEIGHTED SOCIAL GRAPH")
print("=" * 60)

print(
    "Edge index shape:",
    social_edge_index.shape
)

print(
    "Edge weight shape:",
    social_edge_weight.shape
)

print(
    "Weight minimum:",
    social_edge_weight.min().item()
)

print(
    "Weight maximum:",
    social_edge_weight.max().item()
)

print(
    "Weight mean:",
    social_edge_weight.mean().item()
)

WEIGHTED SOCIAL GRAPH
Edge index shape: torch.Size([2, 25434])
Edge weight shape: torch.Size([25434])
Weight minimum: 0.0
Weight maximum: 0.9992053508758545
Weight mean: 0.20653538405895233


In [22]:
#new section 0
N_SEEDS = 3
SEEDS = list(range(N_SEEDS))

K = 10

# Final models: full budget
MAX_EPOCHS = 400
PATIENCE = 60
EVAL_EVERY = 10

# Sweeps: short budget, only used to rank mixing weights
SWEEP_MAX_EPOCHS = 150
SWEEP_PATIENCE = 30

# From the 24-configuration grid search
BEST_EMBEDDING = 128
BEST_LAYERS = 2
BEST_LR = 0.001
BEST_LAMBDA = 1e-5

ALPHA_GRID = [0.0, 0.05, 0.25, 1.0]
BETA_GRID = [0.0, 0.25, 1.0, 2.0, 4.0]

print(f"Seeds: {SEEDS} | max epochs {MAX_EPOCHS} | sweep epochs {SWEEP_MAX_EPOCHS}")

Seeds: [0, 1, 2] | max epochs 400 | sweep epochs 150


In [23]:
# SECTION 1 - SHARED INFRASTRUCTURE
# Seeding, one loss, one evaluator, one trainer

import os
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import wilcoxon


def set_seed(seed: int = 42):
    """Seed every source of randomness."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def bpr_loss(user_emb, pos_emb, neg_emb,
             model, users, pos_items, neg_items,
             lambda_reg=1e-5):
    """
    Bayesian Personalised Ranking loss.

    L2 is applied to the EGO (layer-0) embeddings and normalised by batch
    size, per the LightGCN paper. Using this for every model family is what
    makes the experiments comparable.
    """
    pos_scores = (user_emb * pos_emb).sum(dim=1)
    neg_scores = (user_emb * neg_emb).sum(dim=1)

    ranking_loss = -F.logsigmoid(pos_scores - neg_scores).mean()

    ego_u = model.user_embedding(users)
    ego_p = model.item_embedding(pos_items)
    ego_n = model.item_embedding(neg_items)

    reg = (
        ego_u.norm(2).pow(2)
        + ego_p.norm(2).pow(2)
        + ego_n.norm(2).pow(2)
    ) / (2 * users.shape[0])

    return ranking_loss + lambda_reg * reg


def as_dict(df):
    """user -> set(artist)"""
    return df.groupby("user")["artist"].apply(set).to_dict()


train_dict = as_dict(train_df)
val_dict = as_dict(val_df)
test_dict = as_dict(test_df)


def evaluate_scores(scores, eval_dict, mask_dicts, K=K):
    """
    scores     : anything supporting scores[user] -> 1D array over items
    eval_dict  : ground truth, user -> set(artist)
    mask_dicts : dicts whose items are excluded from ranking

    MASKING RULE (apply without exception):
      validation -> mask_dicts = [train_dict]
      test       -> mask_dicts = [train_dict, val_dict]
    """
    precisions, recalls, hit_rates, ndcgs = [], [], [], []
    per_user_ndcg = {}
    ideal_cache = {}

    for user, truth in eval_dict.items():
        if not truth:
            continue

        row = np.asarray(scores[user], dtype=float).copy()

        for md in mask_dicts:
            masked = md.get(user)
            if masked:
                row[list(masked)] = -np.inf

        k = min(K, row.shape[0])
        top = np.argpartition(row, -k)[-k:]
        top = top[np.argsort(row[top])[::-1]]

        rel = np.fromiter(
            (1.0 if i in truth else 0.0 for i in top),
            dtype=float, count=len(top)
        )
        n_hits = rel.sum()

        precisions.append(n_hits / K)
        recalls.append(n_hits / len(truth))
        hit_rates.append(1.0 if n_hits > 0 else 0.0)

        dcg = float((rel / np.log2(np.arange(len(rel)) + 2)).sum())

        n_truth = min(len(truth), K)
        if n_truth not in ideal_cache:
            ideal_cache[n_truth] = float(
                (1.0 / np.log2(np.arange(n_truth) + 2)).sum()
            )
        idcg = ideal_cache[n_truth]

        nd = dcg / idcg if idcg > 0 else 0.0
        ndcgs.append(nd)
        per_user_ndcg[user] = nd

    return {
        f"Precision@{K}": float(np.mean(precisions)),
        f"Recall@{K}": float(np.mean(recalls)),
        f"HitRate@{K}": float(np.mean(hit_rates)),
        f"NDCG@{K}": float(np.mean(ndcgs)),
        "_per_user_ndcg": per_user_ndcg,
        "_n_users": len(ndcgs),
    }


@torch.no_grad()
def model_scores(model, *forward_args):
    """Dense user x item score matrix."""
    model.eval()
    u, i = model(*forward_args)
    return torch.matmul(u, i.T).cpu().numpy()


def train_with_early_stopping(model_fn, forward_args,
                              learning_rate=BEST_LR,
                              lambda_reg=BEST_LAMBDA,
                              max_epochs=MAX_EPOCHS,
                              patience=PATIENCE,
                              eval_every=EVAL_EVERY,
                              seed=42, verbose=False):
    """
    Trains until validation NDCG@K stops improving, then restores the best
    checkpoint. Every model family gets the same budget and the same
    stopping criterion, which removes 'we trained one longer' as a confound.
    """
    set_seed(seed)

    model = model_fn()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_ndcg = -1.0
    best_state = None
    best_epoch = -1
    stalled = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0

        for users, pos_items, neg_items in train_loader:
            users = users.to(device)
            pos_items = pos_items.to(device)
            neg_items = neg_items.to(device)

            optimizer.zero_grad(set_to_none=True)

            user_emb, item_emb = model(*forward_args)

            loss = bpr_loss(
                user_emb[users], item_emb[pos_items], item_emb[neg_items],
                model, users, pos_items, neg_items,
                lambda_reg=lambda_reg,
            )

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)

        if epoch % eval_every == 0:
            m = evaluate_scores(
                model_scores(model, *forward_args), val_dict, [train_dict]
            )
            ndcg = m[f"NDCG@{K}"]
            history.append((epoch, avg_loss, ndcg))

            if verbose:
                print(f"    epoch {epoch:3d} | loss {avg_loss:.4f} "
                      f"| val NDCG {ndcg:.4f}")

            if ndcg > best_ndcg:
                best_ndcg = ndcg
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                stalled = 0
            else:
                stalled += eval_every
                if stalled >= patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {"best_val_ndcg": best_ndcg,
                   "best_epoch": best_epoch,
                   "history": history}


print("Shared infrastructure ready.")
print(f"Users with test interactions: {len(test_dict)}")

Shared infrastructure ready.
Users with test interactions: 1876


In [24]:
# SECTION 2 - RESULTS REGISTRY
# Every number in the final table comes from here. No literals.


RESULTS = {}          # name -> aggregated metrics
PER_USER = {}         # name -> {user: mean NDCG across seeds}
SEED_TABLES = {}      # name -> DataFrame of per-seed metrics

METRIC_COLS = [f"Precision@{K}", f"Recall@{K}", f"HitRate@{K}", f"NDCG@{K}"]


def record_deterministic(name, scores):
    """For models with no training randomness (popularity, User-CF)."""
    m = evaluate_scores(scores, test_dict, [train_dict, val_dict])
    RESULTS[name] = {c: (m[c], 0.0) for c in METRIC_COLS}
    PER_USER[name] = m["_per_user_ndcg"]
    print(f"  {name:32s} NDCG@{K} = {m[f'NDCG@{K}']:.4f}")
    return m


def record_seeded(name, model_fn_factory, forward_args, seeds=SEEDS, **kw):
    """
    Trains one model per seed, evaluates each on test under the standard
    masking rule, and stores mean +/- std plus seed-averaged per-user NDCG
    for the paired significance tests.
    """
    print(f"\n{name}")
    rows = []
    user_accum = {}

    for s in seeds:
        model, info = train_with_early_stopping(
            model_fn=model_fn_factory,
            forward_args=forward_args,
            seed=s,
            **kw,
        )
        m = evaluate_scores(
            model_scores(model, *forward_args),
            test_dict, [train_dict, val_dict]
        )

        rows.append({"seed": s, "best_epoch": info["best_epoch"],
                     **{c: m[c] for c in METRIC_COLS}})

        for u, nd in m["_per_user_ndcg"].items():
            user_accum.setdefault(u, []).append(nd)

        print(f"  seed {s} | stopped epoch {info['best_epoch']:3d} "
              f"| test NDCG@{K} = {m[f'NDCG@{K}']:.4f}")

    df = pd.DataFrame(rows)
    SEED_TABLES[name] = df
    RESULTS[name] = {c: (df[c].mean(), df[c].std(ddof=1) if len(df) > 1 else 0.0)
                     for c in METRIC_COLS}
    PER_USER[name] = {u: float(np.mean(v)) for u, v in user_accum.items()}

    mu, sd = RESULTS[name][f"NDCG@{K}"]
    print(f"  -> {name}: NDCG@{K} = {mu:.4f} +/- {sd:.4f}")

    return df


print("Registry ready.")

Registry ready.


In [ ]:
# GRID SEARCH — using the shared trainer and evaluator

from itertools import product

embedding_dims = [32, 64, 128]
num_layers_list = [2, 3]
learning_rates = [0.0005, 0.001]
lambda_regs = [1e-5, 1e-4]

results = []
total = (len(embedding_dims) * len(num_layers_list)
         * len(learning_rates) * len(lambda_regs))

print("=" * 70)
print(f"GRID SEARCH ({total} configurations)")
print("=" * 70)

for n, (emb, layers, lr, reg) in enumerate(
    product(embedding_dims, num_layers_list, learning_rates, lambda_regs), 1
):
    print(f"\n[{n}/{total}] emb={emb} layers={layers} lr={lr} lambda={reg}")

    model, info = train_with_early_stopping(
        model_fn=lambda emb=emb, layers=layers: LightGCN(
            num_users=num_users, num_items=num_items,
            embedding_dim=emb, num_layers=layers,
        ).to(device),
        forward_args=(graph.edge_index,),
        learning_rate=lr,
        lambda_reg=reg,
        seed=0,
    )

    m = evaluate_scores(
        model_scores(model, graph.edge_index), val_dict, [train_dict]
    )

    results.append({
        "Embedding": emb, "Layers": layers,
        "Learning Rate": lr, "Lambda": reg,
        "best_epoch": info["best_epoch"],
        **{c: m[c] for c in METRIC_COLS},
    })
    print(f"    val NDCG@10 = {m['NDCG@10']:.4f} "
          f"(stopped epoch {info['best_epoch']})")

results_df = (
    pd.DataFrame(results)
    .sort_values("NDCG@10", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TOP 10 CONFIGURATIONS")
print("=" * 70)
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]
BEST_EMBEDDING = int(best["Embedding"])
BEST_LAYERS = int(best["Layers"])
BEST_LR = float(best["Learning Rate"])
BEST_LAMBDA = float(best["Lambda"])

print(f"\nSelected: emb={BEST_EMBEDDING}, layers={BEST_LAYERS}, "
      f"lr={BEST_LR}, lambda={BEST_LAMBDA}")
results_df.to_csv("grid_search_results.csv", index=False)

GRID SEARCH (24 configurations)

[1/24] emb=32 layers=2 lr=0.0005 lambda=1e-05
    val NDCG@10 = 0.0948 (stopped epoch 400)

[2/24] emb=32 layers=2 lr=0.0005 lambda=0.0001
    val NDCG@10 = 0.0944 (stopped epoch 400)

[3/24] emb=32 layers=2 lr=0.001 lambda=1e-05
    val NDCG@10 = 0.1147 (stopped epoch 400)

[4/24] emb=32 layers=2 lr=0.001 lambda=0.0001
    val NDCG@10 = 0.1131 (stopped epoch 400)

[5/24] emb=32 layers=3 lr=0.0005 lambda=1e-05
    val NDCG@10 = 0.0878 (stopped epoch 400)

[6/24] emb=32 layers=3 lr=0.0005 lambda=0.0001
    val NDCG@10 = 0.0874 (stopped epoch 400)

[7/24] emb=32 layers=3 lr=0.001 lambda=1e-05
    val NDCG@10 = 0.1065 (stopped epoch 400)

[8/24] emb=32 layers=3 lr=0.001 lambda=0.0001
    val NDCG@10 = 0.1052 (stopped epoch 400)

[9/24] emb=64 layers=2 lr=0.0005 lambda=1e-05
    val NDCG@10 = 0.1061 (stopped epoch 400)

[10/24] emb=64 layers=2 lr=0.0005 lambda=0.0001
    val NDCG@10 = 0.1053 (stopped epoch 400)

[11/24] emb=64 layers=2 lr=0.001 lambda=1e-05

In [25]:
# SECTION 3 - NON-GRAPH BASELINES


from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize


class BroadcastScores:
    """Same score vector for every user, without materialising the matrix."""

    def __init__(self, vec):
        self.vec = np.asarray(vec, dtype=float)

    def __getitem__(self, user):
        return self.vec


print("Popularity baselines")

pop_plays = np.zeros(num_items, dtype=float)
np.add.at(pop_plays, train_df["artist"].values,
          train_df["weight"].astype(float).values)

pop_listeners = np.zeros(num_items, dtype=float)
np.add.at(pop_listeners, train_df["artist"].values, 1.0)

record_deterministic("Popularity (plays)", BroadcastScores(pop_plays))
record_deterministic("Popularity (listeners)", BroadcastScores(pop_listeners))



# User-CF with a TUNED neighbourhood size


ui = csr_matrix(
    (train_df["weight"].astype(float).values,
     (train_df["user"].values, train_df["artist"].values)),
    shape=(num_users, num_items),
)

ui_norm = normalize(ui, norm="l2", axis=1)
sim = (ui_norm @ ui_norm.T).toarray()
np.fill_diagonal(sim, 0.0)


def usercf_scores(n_neighbours):
    out = np.zeros((num_users, num_items), dtype=np.float32)
    for u in range(num_users):
        row = sim[u]
        n = min(n_neighbours, int((row > 0).sum()))
        if n == 0:
            continue
        nbrs = np.argpartition(row, -n)[-n:]
        w = row[nbrs]
        out[u] = (ui[nbrs].toarray() * w[:, None]).sum(axis=0)
    return out


print("\nUser-CF neighbourhood tuning (validation)")
cf_rows = []
for n in [5, 10, 20, 40, 80, 160]:
    m = evaluate_scores(usercf_scores(n), val_dict, [train_dict])
    cf_rows.append({"n_neighbours": n, **{c: m[c] for c in METRIC_COLS}})
    print(f"  N={n:3d} | val NDCG@{K} = {m[f'NDCG@{K}']:.4f}")

cf_df = pd.DataFrame(cf_rows).sort_values(f"NDCG@{K}", ascending=False)
BEST_N = int(cf_df.iloc[0]["n_neighbours"])
print(f"  -> best neighbourhood size: {BEST_N}")

_ = record_deterministic(f"User-CF (N={BEST_N})", usercf_scores(BEST_N))

Popularity baselines
  Popularity (plays)               NDCG@10 = 0.0221
  Popularity (listeners)           NDCG@10 = 0.0201

User-CF neighbourhood tuning (validation)
  N=  5 | val NDCG@10 = 0.0609
  N= 10 | val NDCG@10 = 0.0627
  N= 20 | val NDCG@10 = 0.0646
  N= 40 | val NDCG@10 = 0.0693
  N= 80 | val NDCG@10 = 0.0691
  N=160 | val NDCG@10 = 0.0665
  -> best neighbourhood size: 40
  User-CF (N=40)                   NDCG@10 = 0.0702


In [26]:
# ============================================================
# EXPERIMENT E — FINAL SEMANTIC EXTENSION
# Training-only artist-tag graph
# ============================================================


# ------------------------------------------------------------
# 1. BUILD STRICT TRAINING-ONLY ARTIST-TAG GRAPH
# ------------------------------------------------------------

tagsdf = user_taggedartists.copy()

# Map raw IDs using the same mappings as LightGCN
tagsdf["user"] = tagsdf["userID"].map(user_mapping)
tagsdf["artist"] = tagsdf["artistID"].map(artist_mapping)

# Remove tagging events whose user/artist is outside the
# recommendation dataset
tagsdf = tagsdf.dropna(
    subset=["user", "artist"]
).copy()

tagsdf["user"] = tagsdf["user"].astype(int)
tagsdf["artist"] = tagsdf["artist"].astype(int)

print("=" * 60)
print("STRICT TRAINING-ONLY TAG GRAPH")
print("=" * 60)

print(
    "Tag events after mapping:",
    len(tagsdf)
)


# Only retain tagging events whose (user, artist) interaction
# appears in the TRAINING split

train_pairs = set(
    map(
        tuple,
        train_df[["user", "artist"]].values
    )
)

keep_mask = [
    (u, a) in train_pairs
    for u, a in zip(
        tagsdf["user"].values,
        tagsdf["artist"].values
    )
]

tg_strict = tagsdf[keep_mask].copy()

print(
    "Tag events after training restriction:",
    len(tg_strict)
)

print(
    "Retained proportion:",
    f"{len(tg_strict) / len(tagsdf):.1%}"
)


# Unique artist-tag relations

artist_tag_edges = (
    tg_strict[["artist", "tagID"]]
    .drop_duplicates()
    .copy()
)

tag_mapping = {
    tag_id: idx
    for idx, tag_id
    in enumerate(
        artist_tag_edges["tagID"].unique()
    )
}

artist_tag_edges["tag"] = (
    artist_tag_edges["tagID"]
    .map(tag_mapping)
)

num_tags = len(tag_mapping)


# Artist nodes: 0 ... num_items-1
# Tag nodes:    num_items ...

artist_nodes = artist_tag_edges["artist"].values
tag_nodes = (
    artist_tag_edges["tag"].values
    + num_items
)

forward_edges = np.vstack([
    artist_nodes,
    tag_nodes
])

reverse_edges = np.vstack([
    tag_nodes,
    artist_nodes
])

semantic_edge_index = torch.tensor(
    np.hstack([
        forward_edges,
        reverse_edges
    ]),
    dtype=torch.long,
    device=device
)

semantic_edge_index = torch.unique(
    semantic_edge_index,
    dim=1
)


# Identify artists that have semantic information

tagged_mask = torch.zeros(
    num_items,
    dtype=torch.bool,
    device=device
)

tagged_artist_ids = torch.tensor(
    np.sort(
        artist_tag_edges["artist"].unique()
    ),
    dtype=torch.long,
    device=device
)

tagged_mask[tagged_artist_ids] = True

n_tagged = int(tagged_mask.sum().item())
n_untagged = num_items - n_tagged


print("\nArtist-tag edges :", len(artist_tag_edges))
print("Tagged artists   :", n_tagged)
print(
    "Untagged artists :",
    n_untagged,
    f"({n_untagged / num_items:.1%})"
)
print("Distinct tags    :", num_tags)


# ------------------------------------------------------------
# 2. SEMANTIC LIGHTGCN
# ------------------------------------------------------------

class SemanticLightGCN(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        num_tags,
        embedding_dim=128,
        interaction_layers=2,
        semantic_layers=1,
        beta=1.0
    ):
        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.beta = beta

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        self.tag_embedding = nn.Embedding(
            num_tags,
            embedding_dim
        )

        nn.init.xavier_uniform_(
            self.user_embedding.weight
        )

        nn.init.xavier_uniform_(
            self.item_embedding.weight
        )

        nn.init.xavier_uniform_(
            self.tag_embedding.weight
        )

        self.interaction_convs = nn.ModuleList(
            [
                LGConv()
                for _ in range(interaction_layers)
            ]
        )

        self.semantic_convs = nn.ModuleList(
            [
                LGConv()
                for _ in range(semantic_layers)
            ]
        )

    def forward(
        self,
        interaction_edge_index,
        semantic_edge_index
    ):

        users_initial = self.user_embedding.weight
        items_initial = self.item_embedding.weight
        tags_initial = self.tag_embedding.weight


        # Standard LightGCN interaction branch

        x = torch.cat(
            [
                users_initial,
                items_initial
            ],
            dim=0
        )

        embeddings = [x]

        for conv in self.interaction_convs:

            x = conv(
                x,
                interaction_edge_index
            )

            embeddings.append(x)

        interaction_embeddings = torch.stack(
            embeddings,
            dim=1
        ).mean(dim=1)

        interaction_user = (
            interaction_embeddings[
                :self.num_users
            ]
        )

        interaction_item = (
            interaction_embeddings[
                self.num_users:
            ]
        )


        # Artist-tag semantic branch

        semantic_x = torch.cat(
            [
                items_initial,
                tags_initial
            ],
            dim=0
        )

        for conv in self.semantic_convs:

            semantic_x = conv(
                semantic_x,
                semantic_edge_index
            )

        semantic_item = (
            semantic_x[
                :self.num_items
            ]
        )


        # Semantic fusion

        final_item = (
            interaction_item
            + self.beta * semantic_item
        ) / (1.0 + self.beta)

        return interaction_user, final_item


# ------------------------------------------------------------
# 3. MATCHED SHRINKAGE CONTROL
# ------------------------------------------------------------

class ShrinkageControlLightGCN(nn.Module):
    """
    Same fusion structure as SemanticLightGCN, but semantic
    information is replaced by fixed random vectors for tagged
    artists and zero vectors for untagged artists.
    """

    def __init__(
        self,
        num_users,
        num_items,
        tagged_mask,
        embedding_dim=128,
        interaction_layers=2,
        beta=1.0,
        noise_scale=0.01
    ):
        super().__init__()

        self.num_users = num_users
        self.num_items = num_items
        self.beta = beta

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        nn.init.xavier_uniform_(
            self.user_embedding.weight
        )

        nn.init.xavier_uniform_(
            self.item_embedding.weight
        )

        self.interaction_convs = nn.ModuleList(
            [
                LGConv()
                for _ in range(interaction_layers)
            ]
        )

        fake_semantic = (
            torch.randn(
                num_items,
                embedding_dim
            )
            * noise_scale
        )

        fake_semantic[
            ~tagged_mask.cpu()
        ] = 0.0

        self.register_buffer(
            "fake_semantic",
            fake_semantic
        )

    def forward(
        self,
        interaction_edge_index,
        *_ignored
    ):

        users_initial = self.user_embedding.weight
        items_initial = self.item_embedding.weight

        x = torch.cat(
            [
                users_initial,
                items_initial
            ],
            dim=0
        )

        embeddings = [x]

        for conv in self.interaction_convs:

            x = conv(
                x,
                interaction_edge_index
            )

            embeddings.append(x)

        final = torch.stack(
            embeddings,
            dim=1
        ).mean(dim=1)

        interaction_user = (
            final[:self.num_users]
        )

        interaction_item = (
            final[self.num_users:]
        )

        final_item = (
            interaction_item
            + self.beta * self.fake_semantic
        ) / (1.0 + self.beta)

        return interaction_user, final_item


print("\nSemantic and shrinkage-control models defined.")


# ------------------------------------------------------------
# 4. BETA SWEEP — VALIDATION SET, SEED 0
# ------------------------------------------------------------

beta_rows = []

print("\n" + "=" * 60)
print("SEMANTIC BETA SWEEP")
print("=" * 60)

for beta in BETA_GRID:

    model, info = train_with_early_stopping(

        model_fn=lambda beta=beta: SemanticLightGCN(
            num_users=num_users,
            num_items=num_items,
            num_tags=num_tags,
            embedding_dim=BEST_EMBEDDING,
            interaction_layers=BEST_LAYERS,
            semantic_layers=1,
            beta=beta
        ).to(device),

        forward_args=(
            graph.edge_index,
            semantic_edge_index
        ),

        seed=0,
        max_epochs=SWEEP_MAX_EPOCHS,
        patience=SWEEP_PATIENCE
    )

    metrics = evaluate_scores(
        model_scores(
            model,
            graph.edge_index,
            semantic_edge_index
        ),
        val_dict,
        [train_dict]
    )

    beta_rows.append({
        "beta": beta,
        "val_NDCG@10":
            metrics[f"NDCG@{K}"],
        "best_epoch":
            info["best_epoch"]
    })

    print(
        f"beta={beta:<4} | "
        f"val NDCG@{K} = "
        f"{metrics[f'NDCG@{K}']:.4f}"
    )


beta_df = (
    pd.DataFrame(beta_rows)
    .sort_values(
        "val_NDCG@10",
        ascending=False
    )
    .reset_index(drop=True)
)

BEST_BETA = float(
    beta_df.iloc[0]["beta"]
)

print(
    f"\nSelected beta: {BEST_BETA}"
)

beta_df.to_csv(
    "strict_beta_sweep.csv",
    index=False
)

STRICT TRAINING-ONLY TAG GRAPH
Tag events after mapping: 184941
Tag events after training restriction: 59223
Retained proportion: 32.0%

Artist-tag edges : 41884
Tagged artists   : 5711
Untagged artists : 11921 (67.6%)
Distinct tags    : 7009

Semantic and shrinkage-control models defined.

SEMANTIC BETA SWEEP
beta=0.0  | val NDCG@10 = 0.1081
beta=0.25 | val NDCG@10 = 0.1135
beta=1.0  | val NDCG@10 = 0.1137


KeyboardInterrupt: 

In [ ]:
# SECTION 5 - SOCIAL MIXING WEIGHT SWEEPS (validation, seed 0)

def sweep_mixing_weight(label, build_model, forward_args, grid):
    rows = []

    for w in grid:
        model, info = train_with_early_stopping(
            model_fn=lambda w=w: build_model(w),
            forward_args=forward_args,
            seed=0,
            max_epochs=SWEEP_MAX_EPOCHS,
            patience=SWEEP_PATIENCE,
        )

        m = evaluate_scores(
            model_scores(model, *forward_args),
            val_dict,
            [train_dict]
        )

        rows.append({
            "weight": w,
            "best_epoch": info["best_epoch"],
            **{c: m[c] for c in METRIC_COLS}
        })

        print(
            f"  {label}={w:<5} | "
            f"val NDCG@{K} = {m[f'NDCG@{K}']:.4f} "
            f"(epoch {info['best_epoch']})"
        )

    df = pd.DataFrame(rows).sort_values(
        f"NDCG@{K}",
        ascending=False
    )

    best = float(df.iloc[0]["weight"])

    print(f"  -> best {label} = {best}")

    if best == max(grid):
        print(
            f"  !! WARNING: optimum on the grid boundary. Extend it."
        )

    return df, best


print("Social LightGCN - alpha sweep")

alpha_df, BEST_ALPHA = sweep_mixing_weight(
    "alpha",
    lambda a: SocialLightGCN(
        num_users=num_users,
        num_items=num_items,
        embedding_dim=BEST_EMBEDDING,
        interaction_layers=BEST_LAYERS,
        social_layers=1,
        alpha=a,
    ).to(device),
    (graph.edge_index, social_edge_index),
    ALPHA_GRID,
)


print("\nPreference-aware Social LightGCN - alpha sweep")

alpha_d3_df, BEST_ALPHA_D3 = sweep_mixing_weight(
    "alpha",
    lambda a: PreferenceAwareSocialLightGCN(
        num_users=num_users,
        num_items=num_items,
        embedding_dim=BEST_EMBEDDING,
        interaction_layers=BEST_LAYERS,
        social_layers=1,
        alpha=a,
    ).to(device),
    (
        graph.edge_index,
        social_edge_index,
        social_edge_weight
    ),
    ALPHA_GRID,
)

In [ ]:
alpha_df.to_csv(
    "sweep_alpha_social.csv",
    index=False
)

alpha_d3_df.to_csv(
    "sweep_alpha_preference.csv",
    index=False
)

print(
    "Written: sweep_alpha_social.csv, "
    "sweep_alpha_preference.csv"
)

In [ ]:
# SECTION 6 - TRAIN AND EVALUATE EVERY GRAPH MODEL

record_seeded(
    "Tuned LightGCN",
    lambda: LightGCN(
        num_users=num_users,
        num_items=num_items,
        embedding_dim=BEST_EMBEDDING,
        num_layers=BEST_LAYERS,
    ).to(device),
    (graph.edge_index,),
)


record_seeded(
    f"Semantic LightGCN (beta={BEST_BETA})",
    lambda: SemanticLightGCN(
        num_users=num_users,
        num_items=num_items,
        num_tags=num_tags,
        embedding_dim=BEST_EMBEDDING,
        interaction_layers=BEST_LAYERS,
        semantic_layers=1,
        beta=BEST_BETA,
    ).to(device),
    (graph.edge_index, semantic_edge_index),
)


record_seeded(
    f"SHRINKAGE CONTROL (beta={BEST_BETA})",
    lambda: ShrinkageControlLightGCN(
        num_users=num_users,
        num_items=num_items,
        tagged_mask=tagged_mask,
        embedding_dim=BEST_EMBEDDING,
        interaction_layers=BEST_LAYERS,
        beta=BEST_BETA,
    ).to(device),
    (graph.edge_index,),
)


record_seeded(
    f"Social LightGCN (alpha={BEST_ALPHA})",
    lambda: SocialLightGCN(
        num_users=num_users,
        num_items=num_items,
        embedding_dim=BEST_EMBEDDING,
        interaction_layers=BEST_LAYERS,
        social_layers=1,
        alpha=BEST_ALPHA,
    ).to(device),
    (graph.edge_index, social_edge_index),
)


record_seeded(
    f"Preference-Social LightGCN (alpha={BEST_ALPHA_D3})",
    lambda: PreferenceAwareSocialLightGCN(
        num_users=num_users,
        num_items=num_items,
        embedding_dim=BEST_EMBEDDING,
        interaction_layers=BEST_LAYERS,
        social_layers=1,
        alpha=BEST_ALPHA_D3,
    ).to(device),
    (
        graph.edge_index,
        social_edge_index,
        social_edge_weight,
    ),
)

In [ ]:
def final_table():
    rows = []
    for name, metrics in RESULTS.items():
        row = {"Model": name}
        for c in METRIC_COLS:
            mu, sd = metrics[c]
            row[c] = f"{mu:.4f} +/- {sd:.4f}" if sd > 0 else f"{mu:.4f}"
            row[f"_{c}"] = mu
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(f"_NDCG@{K}", ascending=False)
    df.insert(0, "Rank", range(1, len(df) + 1))
    return df.drop(columns=[c for c in df.columns if c.startswith("_")])


print("=" * 100)
print(f"FINAL MODEL COMPARISON - TEST SET ({N_SEEDS} seeds, mean +/- std)")
print("Masking rule: train + validation interactions excluded from ranking")
print("=" * 100)
print(final_table().to_string(index=False))

In [ ]:
def compare_models(name_a, name_b, n_boot=10000, seed=42):
    ndcg_a, ndcg_b = PER_USER[name_a], PER_USER[name_b]
    users = sorted(set(ndcg_a) & set(ndcg_b))

    a = np.array([ndcg_a[u] for u in users])
    b = np.array([ndcg_b[u] for u in users])
    d = a - b

    if np.allclose(d, 0):
        print(f"{name_a} vs {name_b}: identical, no test possible")
        return None

    stat, p = wilcoxon(a, b)

    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])

    rel = 100.0 * d.mean() / b.mean() if b.mean() else float("nan")

    print(f"\n{name_a}")
    print(f"  vs {name_b}      (n = {len(users)} users)")
    print(f"  mean NDCG@{K}      : {a.mean():.4f}  vs  {b.mean():.4f}")
    print(f"  difference        : {d.mean():+.4f}  ({rel:+.2f}%)")
    print(f"  95% bootstrap CI  : [{lo:+.4f}, {hi:+.4f}]")
    print(f"  Wilcoxon p        : {p:.4g}")
    print(f"  users improved    : {(d > 0).sum()} / {len(d)}")
    print(f"  SIGNIFICANT (5%)  : {'YES' if p < 0.05 else 'NO'}")

    if lo <= 0 <= hi:
        print("  NOTE: CI contains zero - not distinguishable from noise")

    return {
        "mean_diff": float(d.mean()),
        "ci": (float(lo), float(hi)),
        "p_value": float(p)
    }


BASE = "Tuned LightGCN"
SEM  = f"Semantic LightGCN (beta={BEST_BETA})"
CTRL = f"SHRINKAGE CONTROL (beta={BEST_BETA})"
SOC  = f"Social LightGCN (alpha={BEST_ALPHA})"
SOC3 = f"Preference-Social LightGCN (alpha={BEST_ALPHA_D3})"

print("=" * 100)
print("PAIRED SIGNIFICANCE TESTS")
print("=" * 100)

sem_vs_base  = compare_models(SEM, BASE)
ctrl_vs_base = compare_models(CTRL, BASE)
sem_vs_ctrl  = compare_models(SEM, CTRL)
soc_vs_base  = compare_models(SOC, BASE)
soc3_vs_base = compare_models(SOC3, BASE)

In [ ]:
# FRIENDSHIP PREFERENCE SIMILARITY FIGURE

fig, ax = plt.subplots(figsize=(4.8, 2.9))

ax.hist(
    friend_similarity,
    bins=50,
    color="#c0504d",
    alpha=0.75,
    edgecolor="white",
    linewidth=0.3
)

med = float(np.median(friend_similarity))

ax.axvline(
    med,
    ls="--",
    c="black",
    lw=1.0
)

ax.text(
    med + 0.02,
    ax.get_ylim()[1] * 0.82,
    f"median = {med:.3f}",
    fontsize=7.5
)

ax.set_xlabel("Cosine similarity of listening histories")
ax.set_ylabel("Number of friendship edges")

plt.tight_layout()

plt.savefig(
    "fig_friendsim.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Median similarity: {med:.6f}")
print(
    f"Zero-similarity edges: "
    f"{int((friend_similarity == 0).sum())} "
    f"({(friend_similarity == 0).mean():.1%})"
)


In [ ]:
# FINAL MODEL COMPARISON FIGURE

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.size": 9,
    "figure.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# Models in the order used for the final comparison

result_names = [
    "Popularity (listeners)",
    "Popularity (plays)",
    f"User-CF (N={BEST_N})",
    f"SHRINKAGE CONTROL (beta={BEST_BETA})",
    "Tuned LightGCN",
    f"Social LightGCN (alpha={BEST_ALPHA})",
    f"Preference-Social LightGCN (alpha={BEST_ALPHA_D3})",
    f"Semantic LightGCN (beta={BEST_BETA})"
]

models = [
    "Popularity\n(listeners)",
    "Popularity\n(plays)",
    f"User-CF\n(N={BEST_N})",
    "Shrinkage\nControl",
    "LightGCN\n(tuned)",
    "Social\nLightGCN",
    "Preference-\nSocial",
    "Semantic\nLightGCN"
]


# Read NDCG means and standard deviations directly from RESULTS

ndcg = [
    RESULTS[name][f"NDCG@{K}"][0]
    for name in result_names
]

err = [
    RESULTS[name][f"NDCG@{K}"][1]
    for name in result_names
]


cols = (
    ["#999999"] * 3
    + ["#c0504d", "#4472c4"]
    + ["#70ad47"] * 3
)


fig, ax = plt.subplots(
    figsize=(7.2, 3.2)
)

ax.bar(
    range(len(models)),
    ndcg,
    yerr=err,
    capsize=3,
    color=cols,
    edgecolor="black",
    linewidth=0.4
)


# Tuned LightGCN reference line

baseline_ndcg = RESULTS[
    "Tuned LightGCN"
][f"NDCG@{K}"][0]

ax.axhline(
    baseline_ndcg,
    ls="--",
    c="#4472c4",
    lw=0.8,
    alpha=0.7
)


for i, (v, e) in enumerate(
    zip(ndcg, err)
):
    ax.text(
        i,
        v + e + 0.004,
        f"{v:.4f}",
        ha="center",
        fontsize=6.8
    )


ax.set_xticks(
    range(len(models))
)

ax.set_xticklabels(
    models,
    fontsize=7.5
)

ax.set_ylabel(
    "NDCG@10 (test)"
)

ax.set_ylim(
    0,
    0.155
)

plt.tight_layout()

plt.savefig(
    "fig_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# MIXING-WEIGHT SWEEP FIGURE

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.7))


# ------------------------------------------------------------
# Semantic beta sweep
# ------------------------------------------------------------

beta_plot = beta_df.sort_values("beta")

b = beta_plot["beta"].values
bv = beta_plot["val_NDCG@10"].values

axes[0].plot(
    b,
    bv,
    "o-",
    color="#70ad47",
    lw=1.6,
    ms=5
)

best_beta_ndcg = beta_plot.loc[
    beta_plot["beta"] == BEST_BETA,
    "val_NDCG@10"
].iloc[0]

axes[0].scatter(
    [BEST_BETA],
    [best_beta_ndcg],
    s=90,
    facecolors="none",
    edgecolors="#c0504d",
    lw=1.6,
    zorder=5
)

axes[0].set_xlabel(
    r"$\beta$ (semantic mixing weight)"
)

axes[0].set_ylabel(
    "Validation NDCG@10"
)

axes[0].set_title(
    "Semantic LightGCN",
    fontsize=9
)


# ------------------------------------------------------------
# Social alpha sweeps
# ------------------------------------------------------------

alpha_plot = alpha_df.sort_values("weight")
alpha_d3_plot = alpha_d3_df.sort_values("weight")

a = alpha_plot["weight"].values

av = alpha_plot[
    f"NDCG@{K}"
].values

av3 = alpha_d3_plot[
    f"NDCG@{K}"
].values


axes[1].plot(
    a,
    av,
    "o-",
    color="#4472c4",
    lw=1.6,
    ms=5,
    label="Unweighted"
)

axes[1].plot(
    a,
    av3,
    "s-",
    color="#c0504d",
    lw=1.6,
    ms=5,
    label="Preference-weighted"
)

axes[1].set_xlabel(
    r"$\alpha$ (social mixing weight)"
)

axes[1].set_ylabel(
    "Validation NDCG@10"
)

axes[1].set_title(
    "Social LightGCN",
    fontsize=9
)

axes[1].legend(
    fontsize=7.5,
    frameon=False
)


plt.tight_layout()

plt.savefig(
    "fig_sweeps.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# PAIRED SIGNIFICANCE TEST FIGURE

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.size": 9,
    "figure.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False
})


labels = [
    "Semantic vs\nLightGCN",
    "Preference-Social vs\nLightGCN",
    "Social vs\nLightGCN",
    "Shrinkage Control vs\nLightGCN",
    "Semantic vs\nShrinkage Control"
]

comparisons = [
    sem_vs_base,
    soc3_vs_base,
    soc_vs_base,
    ctrl_vs_base,
    sem_vs_ctrl
]


diff = [
    r["mean_diff"]
    for r in comparisons
]

lo = [
    r["ci"][0]
    for r in comparisons
]

hi = [
    r["ci"][1]
    for r in comparisons
]

sig = [
    r["p_value"] < 0.05
    for r in comparisons
]


fig, ax = plt.subplots(
    figsize=(5.6, 2.9)
)

y = np.arange(len(labels))

for i in range(len(labels)):

    if sig[i] and diff[i] > 0:
        c = "#70ad47"
    elif sig[i] and diff[i] < 0:
        c = "#c0504d"
    else:
        c = "#999999"

    ax.plot(
        [lo[i], hi[i]],
        [y[i], y[i]],
        c=c,
        lw=2.2
    )

    ax.plot(
        diff[i],
        y[i],
        "o",
        c=c,
        ms=6
    )


ax.axvline(
    0,
    ls="--",
    c="black",
    lw=0.9
)

ax.set_yticks(y)

ax.set_yticklabels(
    labels,
    fontsize=7.5
)

ax.invert_yaxis()

ax.set_xlabel(
    "Difference in mean NDCG@10 (95% bootstrap CI)"
)

plt.tight_layout()

plt.savefig(
    "fig_sig.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ua = pd.read_csv("user_artists.dat", sep="\t")

listeners = (
    ua.groupby("artistID")["userID"]
    .nunique()
    .sort_values(ascending=False)
    .values
)

plt.rcParams.update({
    "font.size": 9,
    "figure.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False
})

fig, ax = plt.subplots(figsize=(4.8, 2.9))

ax.loglog(
    np.arange(1, len(listeners) + 1),
    listeners,
    color="#4472c4",
    lw=1.4
)

ax.set_xlabel("Artist rank by number of listeners")
ax.set_ylabel("Number of distinct listeners")

plt.tight_layout()

plt.savefig(
    "fig_longtail.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()